# Session 0 — Prepare both tutorial datasets

## Goal

Run this notebook once before the workshop. It prepares two deliberately separate examples:

- **Sessions 1–2: Tonsil**, with paired spatial RNA and measured ADT protein data for preprocessing, graph construction, and DGAT teaching.
- **Session 3: Human lymph node**, a transcript-only Visium sample with GC annotations and organizer-validated, precomputed DGAT protein predictions.

The default participant workflow loads the lymph-node prediction matrix and is CPU-friendly. Released model weights are downloaded only when the optional full-inference section in Session 3 is enabled.


## 1. Mount Drive and fetch the tutorial repository (estimated runtime: 1 min)


In [ ]:
from pathlib import Path
import importlib.util, os, subprocess, sys
if importlib.util.find_spec("google.colab") is None: raise RuntimeError("Open Session 0 in Google Colab.")
from google.colab import drive
drive.mount("/content/drive", force_remount=False)
repo_dir = Path("/content/ECCB-2026-Tutorial"); tutorial_root = repo_dir / "hands-on_tutorial"
if not (tutorial_root / "src" / "dgat_tutorial").is_dir():
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/osmanbeyoglulab/ECCB-2026-Tutorial.git", str(repo_dir)], check=True)
else:
    subprocess.run(["git", "-C", str(repo_dir), "fetch", "--depth", "1", "origin", "main"], check=True)
    subprocess.run(["git", "-C", str(repo_dir), "reset", "--hard", "origin/main"], check=True)
os.chdir(tutorial_root); print("Tutorial repository:", tutorial_root)


## 2. Download and verify both dataset families (estimated runtime: 5–10 min)

This step downloads approximately 400 MB: paired Tonsil H5AD files plus the public 10x lymph-node matrix and spatial image. Existing complete files are reused. The validated lymph-node predictions and GC labels are copied from the tracked tutorial assets.


In [ ]:
drive_root = Path("/content/drive/MyDrive/ECCB2026")
asset_root = drive_root / "assets" / "DGAT_assets"
asset_root.mkdir(parents=True, exist_ok=True)

if importlib.util.find_spec("gdown") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gdown==6.1.0"], check=True)

organizer_output = drive_root / "organizer" / "lymph_node_prediction_build"
environment = os.environ.copy()
environment["DGAT_ASSET_DIR"] = str(asset_root)
environment["DGAT_PRECOMPUTED_DIR"] = str(organizer_output)
command = ["bash", "scripts/download_dgat_assets.sh", "--dataset", "all"]
subprocess.run(command, cwd=tutorial_root, env=environment, check=True)
subprocess.run(command + ["--check-only"], cwd=tutorial_root, env=environment, check=True)


## 3. Record a combined checksum manifest (estimated runtime: 1 min)

Every later notebook verifies the byte size of only the files it needs. Session 3 additionally validates prediction provenance, checksum, barcode order, and the ordered 31-protein contract before analysis.


In [ ]:
import hashlib, json

data_root = asset_root / "data"
files = [
    data_root / "Tonsil_RNA.h5ad",
    data_root / "Tonsil_ADT.h5ad",
    data_root / "V1_Human_Lymph_Node_filtered_feature_bc_matrix.h5",
    data_root / "V1_Human_Lymph_Node_spatial.tar.gz",
    data_root / "V1_Human_Lymph_Node_manual_GC_annot.csv",
    data_root / "V1_Human_Lymph_Node_DGAT_predicted_proteins.csv",
    data_root / "V1_Human_Lymph_Node_DGAT_predicted_proteins.metadata.json",
    data_root / "spatial/tissue_positions_list.csv",
    data_root / "spatial/tissue_hires_image.png",
]
manifest = {
    "workflow": "mixed_tonsil_lymph_node",
    "python": sys.version.split()[0],
    "datasets": {
        "Tonsil": {"sessions": [1, 2], "modalities": ["RNA", "measured ADT"]},
        "V1_Human_Lymph_Node": {
            "sessions": [3],
            "modalities": ["RNA", "spatial image", "GC annotation", "predicted protein"],
            "evaluation_scope": "indirect biological evaluation; no measured protein ground truth",
        },
    },
    "files": {},
}
for path in files:
    if not path.is_file() or path.stat().st_size == 0:
        raise FileNotFoundError(path)
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(8 * 1024 * 1024), b""):
            digest.update(chunk)
    key = str(path.relative_to(data_root))
    manifest["files"][key] = {"bytes": path.stat().st_size, "sha256": digest.hexdigest()}

manifest_path = drive_root / "asset_manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2) + "\n")
print(json.dumps(manifest, indent=2))


## 4. Cache the environment and create restart-safe folders (estimated runtime: 1–3 min)

A free-tier CPU runtime is sufficient for the default workflow. Allow at least **3 GB of free Drive space** for source files, cached wheels, and the large processed Tonsil matrices. The optional lymph-node inference rerun is better suited to a GPU runtime and is not required during the hands-on session.


In [ ]:
wheelhouse = drive_root / "wheelhouse" / f"py{sys.version_info.major}{sys.version_info.minor}"; wheelhouse.mkdir(parents=True, exist_ok=True)
subprocess.run([sys.executable, "-m", "pip", "download", "-q", "--only-binary=:all:", "--dest", str(wheelhouse), "-r", str(tutorial_root / "requirements-colab.txt")], check=True)
for relative in ("state/data/processed", "state/results/figures", "state/checkpoints"): (drive_root / relative).mkdir(parents=True, exist_ok=True)
print("Drive preparation complete:", drive_root)
